# Silver project-to-province-reporting-area map

## Purpose

Publish one governed D-57 reporting-area result for every canonical project. The table name retains `province_map` for discoverability, but targets may be an official province, an independent-city reporting area, NCR, or Unknown.

## Grain

One canonical project per selected project snapshot, PSGC run, boundary run, approved config version, and mapping-rule version. Candidate fan-out is aggregated before publication.


## 1. Gate inputs and select governed versions


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

DECLARE OR REPLACE VARIABLE mapping_rule_version STRING
    DEFAULT 'silver-project-province-reporting-area-map-v1';
DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
DECLARE OR REPLACE VARIABLE min_latitude DOUBLE DEFAULT 4.2;
DECLARE OR REPLACE VARIABLE max_latitude DOUBLE DEFAULT 21.3;
DECLARE OR REPLACE VARIABLE min_longitude DOUBLE DEFAULT 116.0;
DECLARE OR REPLACE VARIABLE max_longitude DOUBLE DEFAULT 127.0;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: silver_project is empty, mixed-run, or duplicated'
)
FROM `02-silver`.silver_project;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT reporting_area_key)
    AND COUNT_IF(reporting_area_key = 0) = 1,
    'STOP: reporting-area reference is empty, mixed-run, duplicated, or lacks Unknown'
)
FROM `02-silver`.silver_province_reporting_area;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_province_reporting_area)
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT reporting_area_key),
    'STOP: reporting-area boundary result is incomplete or duplicated'
)
FROM `02-silver`.silver_province_reporting_area_boundary;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT alias_version) <= 1,
    'STOP: multiple active approved DPWH province-alias versions are unsafe'
)
FROM `02-silver`.config_place_name_alias
WHERE source_system = 'dpwh_projects'
  AND UPPER(TRIM(place_type)) IN ('PROVINCE', 'CITY', 'REGION')
  AND UPPER(TRIM(match_rule)) = 'DPWH_PROVINCE_REPORTING_AREA'
  AND approval_status = 'APPROVED'
  AND is_active;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT mapping_version) <= 1,
    'STOP: multiple active approved DPWH project-override versions are unsafe'
)
FROM `02-silver`.config_manual_geographic_match
WHERE source_system = 'DPWH'
  AND UPPER(TRIM(record_type)) = 'PROJECT'
  AND approval_status = 'APPROVED'
  AND is_active;

WITH governed_alias AS (
    SELECT alias.*, area.reporting_area_key, area.reporting_area_type
    FROM `02-silver`.config_place_name_alias AS alias
    LEFT JOIN `02-silver`.silver_province_reporting_area AS area
        ON alias.canonical_psgc_code = area.psgc_reporting_area_code
    WHERE alias.source_system = 'dpwh_projects'
      AND UPPER(TRIM(alias.match_rule)) = 'DPWH_PROVINCE_REPORTING_AREA'
      AND alias.approval_status = 'APPROVED' AND alias.is_active
)
SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'source', REGEXP_REPLACE(UPPER(TRIM(raw_place_name)), '\\s+', ' '),
        'version', alias_version
    ))
    AND COUNT_IF(TRIM(COALESCE(raw_place_name, '')) = ''
        OR UPPER(TRIM(raw_place_name)) IN ('UNKNOWN', 'N/A', 'NA', 'NONE', '-')) = 0
    AND COUNT_IF(reporting_area_key IS NULL) = 0
    AND COUNT_IF(
        (UPPER(TRIM(place_type)) = 'PROVINCE' AND reporting_area_type <> 'PROVINCE')
        OR (UPPER(TRIM(place_type)) = 'CITY' AND reporting_area_type <> 'INDEPENDENT_CITY')
        OR (UPPER(TRIM(place_type)) = 'REGION' AND reporting_area_type <> 'NCR')
    ) = 0,
    'STOP: approved project reporting-area aliases are ambiguous, generic, or invalid'
)
FROM governed_alias;

WITH governed_manual AS (
    SELECT manual.*, area.reporting_area_key
    FROM `02-silver`.config_manual_geographic_match AS manual
    LEFT JOIN `02-silver`.silver_province_reporting_area AS area
        ON manual.canonical_psgc_code = area.psgc_reporting_area_code
    WHERE manual.source_system = 'DPWH'
      AND UPPER(TRIM(manual.record_type)) = 'PROJECT'
      AND manual.approval_status = 'APPROVED' AND manual.is_active
)
SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'project', source_record_id, 'version', mapping_version
    ))
    AND COUNT_IF(TRIM(COALESCE(source_record_id, '')) = '') = 0
    AND COUNT_IF(reporting_area_key IS NULL) = 0,
    'STOP: approved manual project reporting-area targets are ambiguous or invalid'
)
FROM governed_manual;

CREATE OR REPLACE TEMPORARY VIEW selected_project_province_versions AS
WITH project AS (
    SELECT
        MAX(run_id) AS project_run_id,
        MAX(source_snapshot_id) AS project_source_snapshot_id,
        MAX(source_ingest_run_id) AS project_ingest_run_id,
        MAX(source_modified_at) AS project_source_load_ts
    FROM `02-silver`.silver_project
),
area AS (
    SELECT
        MAX(run_id) AS reporting_area_run_id,
        MAX(psgc_run_id) AS psgc_run_id,
        MAX(psgc_version) AS psgc_version,
        MAX(source_load_ts) AS psgc_source_load_ts
    FROM `02-silver`.silver_province_reporting_area
),
boundary AS (
    SELECT
        MAX(run_id) AS reporting_area_boundary_run_id,
        MAX(boundary_version) AS boundary_version,
        MAX(boundary_run_id) AS boundary_run_id,
        MAX(source_load_ts) AS boundary_source_load_ts
    FROM `02-silver`.silver_province_reporting_area_boundary
),
config AS (
    SELECT
        COALESCE((
            SELECT MAX(alias_version)
            FROM `02-silver`.config_place_name_alias
            WHERE source_system = 'dpwh_projects'
              AND UPPER(TRIM(place_type)) IN ('PROVINCE', 'CITY', 'REGION')
              AND UPPER(TRIM(match_rule)) = 'DPWH_PROVINCE_REPORTING_AREA'
              AND approval_status = 'APPROVED' AND is_active
        ), 'NO_APPROVED_PROVINCE_ALIAS') AS alias_version,
        COALESCE((
            SELECT MAX(mapping_version)
            FROM `02-silver`.config_manual_geographic_match
            WHERE source_system = 'DPWH'
              AND UPPER(TRIM(record_type)) = 'PROJECT'
              AND approval_status = 'APPROVED' AND is_active
        ), 'NO_APPROVED_MANUAL_EXCEPTION') AS manual_mapping_version
)
SELECT
    project.*, area.*, boundary.*, config.*,
    CONCAT(config.alias_version, '|', config.manual_mapping_version)
        AS mapping_config_version,
    SHA2(CONCAT_WS(
        '||', project.project_source_snapshot_id, project.project_run_id,
        project.project_ingest_run_id, area.psgc_run_id,
        COALESCE(area.psgc_version, 'NO_PUBLISHER_VERSION'),
        boundary.reporting_area_boundary_run_id,
        COALESCE(boundary.boundary_version, 'NO_BOUNDARY_VERSION'),
        boundary.boundary_run_id, config.alias_version,
        config.manual_mapping_version, mapping_rule_version
    ), 256) AS mapping_run_id
FROM project
CROSS JOIN area
CROSS JOIN boundary
CROSS JOIN config;


## 2. Prepare source, config, exact-name, and boundary candidates

Raw `location.province` is source evidence only. Exact official-name candidates remain diagnostic unless an active approved config row authorizes the value. Spatial matching uses only valid governed boundaries, `ST_COVERS`, and longitude-first EPSG:4326 points.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_province_input AS
WITH raw_source AS (
    SELECT
        project.project_key,
        NULLIF(TRIM(GET_JSON_OBJECT(bronze.source_record_json, '$.location.province')), '')
            AS reported_province_raw,
        REGEXP_REPLACE(UPPER(NULLIF(TRIM(GET_JSON_OBJECT(
            bronze.source_record_json, '$.location.province'
        )), '')), '\\s+', ' ') AS reported_province_standardized
    FROM `02-silver`.silver_project AS project
    INNER JOIN `01-bronze`.dpwh_projects AS bronze
        ON project.contract_id = bronze.contract_id
       AND project.source_snapshot_id = bronze._source_snapshot_id
),
source_rollup AS (
    SELECT
        project_key,
        COUNT(DISTINCT reported_province_standardized) AS source_province_value_count,
        CASE WHEN COUNT(DISTINCT reported_province_standardized) = 1
            THEN MAX(reported_province_raw) END AS reported_province_raw,
        CASE WHEN COUNT(DISTINCT reported_province_standardized) = 1
            THEN MAX(reported_province_standardized) END
            AS reported_province_standardized
    FROM raw_source
    GROUP BY project_key
)
SELECT
    project.project_key, project.contract_id, project.reported_region,
    REGEXP_REPLACE(UPPER(TRIM(project.reported_region)), '\\s+', ' ')
        AS reported_region_standardized,
    source.reported_province_raw, source.reported_province_standardized,
    COALESCE(source.source_province_value_count, 0) AS source_province_value_count,
    project.latitude_parsed AS latitude, project.longitude_parsed AS longitude,
    CASE
        WHEN project.coordinate_resolution_status <> 'RESOLVED' THEN 'MISSING_OR_CONFLICTING'
        WHEN project.latitude_parsed IS NULL OR project.longitude_parsed IS NULL
            THEN 'MISSING_OR_CONFLICTING'
        WHEN project.latitude_parsed NOT BETWEEN min_latitude AND max_latitude
          OR project.longitude_parsed NOT BETWEEN min_longitude AND max_longitude
            THEN 'OUT_OF_RANGE'
        ELSE 'VALID_PAIR'
    END AS coordinate_status,
    project.reported_budget_pesos, project.source_snapshot_id,
    project.source_ingest_run_id, project.run_id AS project_run_id
FROM `02-silver`.silver_project AS project
LEFT JOIN source_rollup AS source ON project.project_key = source.project_key;

CREATE OR REPLACE TEMPORARY VIEW approved_project_province_aliases AS
SELECT
    REGEXP_REPLACE(UPPER(TRIM(alias.raw_place_name)), '\\s+', ' ')
        AS raw_province_standardized,
    area.reporting_area_key, area.psgc_reporting_area_code,
    alias.alias_version
FROM `02-silver`.config_place_name_alias AS alias
INNER JOIN `02-silver`.silver_province_reporting_area AS area
    ON alias.canonical_psgc_code = area.psgc_reporting_area_code
CROSS JOIN selected_project_province_versions AS selected
WHERE alias.source_system = 'dpwh_projects'
  AND UPPER(TRIM(alias.place_type)) IN ('PROVINCE', 'CITY', 'REGION')
  AND UPPER(TRIM(alias.match_rule)) = 'DPWH_PROVINCE_REPORTING_AREA'
  AND COALESCE(TRIM(alias.raw_region_name), '') = ''
  AND COALESCE(TRIM(alias.raw_province_name), '') = ''
  AND COALESCE(TRIM(alias.raw_city_municipality_name), '') = ''
  AND alias.approval_status = 'APPROVED' AND alias.is_active
  AND alias.alias_version = selected.alias_version;

CREATE OR REPLACE TEMPORARY VIEW approved_project_province_overrides AS
SELECT
    manual.source_record_id AS contract_id, area.reporting_area_key,
    area.psgc_reporting_area_code, manual.override_reason, manual.mapping_version
FROM `02-silver`.config_manual_geographic_match AS manual
INNER JOIN `02-silver`.silver_province_reporting_area AS area
    ON manual.canonical_psgc_code = area.psgc_reporting_area_code
CROSS JOIN selected_project_province_versions AS selected
WHERE manual.source_system = 'DPWH'
  AND UPPER(TRIM(manual.record_type)) = 'PROJECT'
  AND manual.approval_status = 'APPROVED' AND manual.is_active
  AND manual.mapping_version = selected.manual_mapping_version;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT raw_province_standardized),
    'STOP: one approved source province value maps to multiple reporting areas'
)
FROM approved_project_province_aliases;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT contract_id),
    'STOP: one project has multiple approved manual reporting-area targets'
)
FROM approved_project_province_overrides;

CREATE OR REPLACE TEMPORARY VIEW exact_reporting_area_name_candidates AS
SELECT
    project.project_key,
    COUNT(DISTINCT area.reporting_area_key) AS exact_name_candidate_count,
    CASE WHEN COUNT(DISTINCT area.reporting_area_key) = 1
        THEN MAX(area.reporting_area_key) END AS exact_name_reporting_area_key
FROM project_province_input AS project
INNER JOIN `02-silver`.silver_province_reporting_area AS area
    ON project.reported_province_standardized
        = REGEXP_REPLACE(UPPER(TRIM(area.reporting_area_name)), '\\s+', ' ')
WHERE project.source_province_value_count = 1
  AND area.reporting_area_type <> 'UNKNOWN'
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW approved_alias_candidates AS
SELECT
    project.project_key,
    COUNT(DISTINCT alias.reporting_area_key) AS alias_candidate_count,
    CASE WHEN COUNT(DISTINCT alias.reporting_area_key) = 1
        THEN MAX(alias.reporting_area_key) END AS alias_reporting_area_key
FROM project_province_input AS project
INNER JOIN approved_project_province_aliases AS alias
    ON project.reported_province_standardized = alias.raw_province_standardized
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW spatial_reporting_area_candidates AS
SELECT /*+ BROADCAST(boundary) */
    project.project_key,
    COUNT(DISTINCT boundary.reporting_area_key) AS spatial_candidate_count,
    CASE WHEN COUNT(DISTINCT boundary.reporting_area_key) = 1
        THEN MAX(boundary.reporting_area_key) END AS spatial_reporting_area_key
FROM project_province_input AS project
INNER JOIN `02-silver`.silver_province_reporting_area_boundary AS boundary
    ON ST_COVERS(
        ST_SETSRID(ST_GEOMFROMWKB(boundary.boundary_geometry_wkb), source_geometry_srid),
        ST_POINT(project.longitude, project.latitude, source_geometry_srid)
    )
WHERE project.coordinate_status = 'VALID_PAIR'
  AND boundary.boundary_status = 'VALID_BOUNDARY'
GROUP BY project.project_key;


## 3. Apply approved precedence and publish one result

Precedence is non-geographic, approved manual exception, project PSGC evidence when available, approved alias, unique coordinate containment, then Unknown. The current project contract has no legitimate project PSGC field, so that stage is explicitly unavailable. Exact official source names remain diagnostic and cannot authorize a match by themselves.


In [0]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_project_province_map
USING DELTA
COMMENT 'One governed D-57 province-reporting-area result per canonical project. The target can be a province, independent city, NCR, or Unknown.'
AS
WITH evidence AS (
    SELECT
        project.*,
        manual.reporting_area_key AS manual_reporting_area_key,
        manual.override_reason AS manual_override_reason,
        COALESCE(exact.exact_name_candidate_count, 0) AS exact_name_candidate_count,
        exact.exact_name_reporting_area_key,
        COALESCE(alias.alias_candidate_count, 0) AS alias_candidate_count,
        alias.alias_reporting_area_key,
        COALESCE(spatial.spatial_candidate_count, 0) AS spatial_candidate_count,
        spatial.spatial_reporting_area_key
    FROM project_province_input AS project
    LEFT JOIN approved_project_province_overrides AS manual
        ON project.contract_id = manual.contract_id
    LEFT JOIN exact_reporting_area_name_candidates AS exact
        ON project.project_key = exact.project_key
    LEFT JOIN approved_alias_candidates AS alias
        ON project.project_key = alias.project_key
    LEFT JOIN spatial_reporting_area_candidates AS spatial
        ON project.project_key = spatial.project_key
),
decision AS (
    SELECT
        evidence.*,
        CASE
            WHEN reported_region_standardized = 'CENTRAL OFFICE' THEN CAST(0 AS BIGINT)
            WHEN manual_reporting_area_key IS NOT NULL THEN manual_reporting_area_key
            WHEN alias_candidate_count = 1 THEN alias_reporting_area_key
            WHEN spatial_candidate_count = 1 THEN spatial_reporting_area_key
            ELSE CAST(0 AS BIGINT)
        END AS accepted_reporting_area_key,
        CASE
            WHEN reported_region_standardized = 'CENTRAL OFFICE' THEN 'NON_GEOGRAPHIC'
            WHEN manual_reporting_area_key IS NOT NULL THEN 'MANUAL_EXCEPTION'
            WHEN alias_candidate_count = 1 THEN 'APPROVED_ALIAS'
            WHEN spatial_candidate_count = 1 THEN 'COORDINATE_BOUNDARY'
            ELSE 'NONE'
        END AS mapping_method
    FROM evidence
),
classified AS (
    SELECT
        decision.*,
        (
            accepted_reporting_area_key <> 0
            AND (
                (exact_name_candidate_count = 1
                 AND exact_name_reporting_area_key <> accepted_reporting_area_key)
                OR (spatial_candidate_count = 1
                    AND spatial_reporting_area_key <> accepted_reporting_area_key)
                OR spatial_candidate_count > 1
            )
        ) AS conflict_flag
    FROM decision
)
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'PROJECT_PROVINCE_REPORTING_AREA_MAP',
        CAST(classified.project_key AS STRING), selected.mapping_run_id
    )) AS project_province_map_key,
    classified.project_key, classified.contract_id,
    classified.reported_province_raw, classified.reported_province_standardized,
    classified.latitude, classified.longitude, classified.coordinate_status,
    classified.accepted_reporting_area_key AS reporting_area_key,
    area.psgc_reporting_area_code, area.reporting_area_name, area.reporting_area_type,
    area.parent_region_code, area.parent_region_name,
    classified.mapping_method,
    CASE
        WHEN classified.mapping_method = 'NON_GEOGRAPHIC' THEN 'NON_GEOGRAPHIC'
        WHEN classified.accepted_reporting_area_key <> 0 AND classified.conflict_flag
            THEN 'MATCHED_WITH_CONFLICT'
        WHEN classified.accepted_reporting_area_key <> 0 THEN 'MATCHED'
        WHEN classified.spatial_candidate_count > 1
          OR classified.alias_candidate_count > 1 THEN 'AMBIGUOUS'
        WHEN classified.source_province_value_count > 1 THEN 'INVALID_SOURCE'
        ELSE 'UNMAPPED'
    END AS match_status,
    CASE classified.mapping_method
        WHEN 'MANUAL_EXCEPTION' THEN 'REVIEWED'
        WHEN 'APPROVED_ALIAS' THEN 'REVIEWED_EXACT'
        WHEN 'COORDINATE_BOUNDARY' THEN 'SPATIAL_UNIQUE'
        WHEN 'NON_GEOGRAPHIC' THEN 'NOT_APPLICABLE'
        ELSE 'UNRESOLVED'
    END AS match_quality,
    CASE
        WHEN classified.mapping_method = 'NON_GEOGRAPHIC'
            THEN 'Central Office is not a geographic reporting area'
        WHEN classified.mapping_method = 'MANUAL_EXCEPTION'
            THEN classified.manual_override_reason
        WHEN classified.mapping_method = 'APPROVED_ALIAS'
            THEN 'Active approved exact source-name alias'
        WHEN classified.mapping_method = 'COORDINATE_BOUNDARY'
            THEN 'One valid reporting-area boundary covers the project point'
        WHEN classified.spatial_candidate_count > 1
            THEN 'More than one valid reporting-area boundary covers the point'
        WHEN classified.coordinate_status <> 'VALID_PAIR'
            THEN 'No usable coordinate and no stronger approved evidence'
        ELSE 'No approved evidence resolved one reporting area'
    END AS mapping_reason,
    GREATEST(classified.exact_name_candidate_count, classified.alias_candidate_count)
        AS name_candidate_count,
    classified.spatial_candidate_count, classified.conflict_flag,
    classified.source_snapshot_id AS project_source_snapshot_id,
    classified.project_run_id, classified.source_ingest_run_id AS project_ingest_run_id,
    selected.psgc_version, selected.psgc_run_id,
    selected.boundary_version, selected.boundary_run_id,
    selected.mapping_config_version, mapping_rule_version,
    selected.mapping_run_id AS run_id,
    GREATEST(
        selected.project_source_load_ts, selected.psgc_source_load_ts,
        selected.boundary_source_load_ts
    ) AS source_load_ts
FROM classified
CROSS JOIN selected_project_province_versions AS selected
INNER JOIN `02-silver`.silver_province_reporting_area AS area
    ON classified.accepted_reporting_area_key = area.reporting_area_key;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_project)
    AND COUNT(*) = COUNT(DISTINCT map.project_key)
    AND COUNT(*) = COUNT(DISTINCT project_province_map_key)
    AND COUNT_IF(project_province_map_key <> XXHASH64(CONCAT_WS(
        '|', 'PROJECT_PROVINCE_REPORTING_AREA_MAP',
        CAST(map.project_key AS STRING), map.run_id
    ))) = 0
    AND COUNT(DISTINCT map.run_id) = 1
    AND MAX(map.run_id) = (SELECT mapping_run_id FROM selected_project_province_versions)
    AND SUM(project.reported_budget_pesos) <=> (
        SELECT SUM(reported_budget_pesos) FROM `02-silver`.silver_project
    ),
    'STOP: project reporting-area mapping changed project grain or budget total'
)
FROM `02-silver`.silver_project_province_map AS map
INNER JOIN `02-silver`.silver_project AS project
    ON map.project_key = project.project_key;


# Summary

The production map keeps every canonical project once and sends unresolved, ambiguous, invalid, and Central Office records to governed key 0 with the reason preserved. Exact raw DPWH province text never authorizes a target without active approved config. Databricks runtime evidence is still required.
